# Lesson 81 · A music recommender — it turns out to be an arithmetic problem: average the embeddings, add k-NN, and write a playlist that gets you

**Goal**: squeeze "which songs you like" into a single profile vector, then use k-NN to build a recommended playlist.

> **In plain words**: content-based recommendation (like what you've already heard) vs collaborative filtering (like what other people liked); cold start = what to do when someone hasn't listened to anything yet.

🔗 **Aurora connection**: this lesson is a core deliverable of the Aurora Music Intelligence Engine. The recommendation logic maps to `src/aurora/music/similarity.py` (`find_similar`); song feature extraction maps to `src/aurora/music/features.py`; song embeddings map to `src/aurora/music/embed.py` (L79's `MusicEncoder`, already in place). (`recommend.py` / `sim.py` are planned stub files and have not been created yet.)

← **Previous lesson** · [L80 · Vector similarity search](L80_similarity.ipynb)

> Last lesson covered **vector similarity search**: cosine similarity vs dot product vs L2, with a pure-NumPy k-NN.
> This lesson explores **a music recommender**.

## Learning goals

By the end of this lesson you should be able to:

1. **Implement a mean-embedding user profile**: compute the normalized average embedding from a user's list of history ids.
2. **Handle the cold-start problem**: when the history is empty, fall back to the global mean embedding as a neutral starting point.
3. **Implement the filtering logic**: correctly exclude songs already in the history from the recommendations.
4. **Assess how embedding quality affects Precision@K**: run an experiment that controls a noise coefficient, and understand the relationship between embedding quality and recommendation precision.

> **Aurora connection**: `aurora.music.similarity` (similarity search), `aurora.music.features` (feature extraction)

## This lesson's story: Netflix's recommender looks mysterious, but in code it comes down to one main thread — average the songs you like into a single vector.

Netflix has 200 million users, and each of them watches different things every day. Its recommendation algorithm sounds mysterious, but the core idea is remarkably simple:

**The average of the embedding vectors of the movies/songs you like is the vector for "you".**

```
User history: "Harry Potter" embedding  +  "The Lord of the Rings" embedding  +  "Narnia" embedding
              ──────────────────────────────────────────────────────────
                           ÷ 3 (average)
                               ↓
                      user_embedding (a fantasy-fan vector)
                               ↓
                      find the nearest top-10 songs/movies
                               ↓
                      recommendation list (with already-watched ones filtered out)
```

**Three design questions**:

| Problem | Solution |
|---|---|
| The user history is empty (a new user) | Cold start: use the library-wide mean, or the popularity chart |
| History songs show up in the recommendations | Filtering: remove already-heard songs from the top-k candidates |
| The user's taste drifts (changes over time) | Weighted average: recent history gets more weight |

**Precision@k**: a metric for recommendation quality. If you validate against a ground truth of "songs the user will go on to like", Precision@5 = how many of the 5 recommended songs are actually liked / 5.

This lesson's task: implement `recommend()`, chaining history → average embedding → filtering → top-k into one complete recommendation function.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

# If mu3_sim is implemented, uncomment:
# from aurora.music.similarity import find_similar  # correct namespace (old: month04.mu3_sim is deprecated)

# A self-contained version for this lesson: similarity search implemented directly with numpy
np.random.seed(42)


## 1. User profile = mean(liked song embeddings)

Given the list of ids of songs the user likes, `history = [i₀, i₁, ..., iₙ]`, take the matching rows from the full embedding matrix `all_embs` (shape `[N, d]`) and average them:

```
user_emb = all_embs[history].mean(axis=0)   # shape: (d,)
```

Then L2-normalize it so its length is 1; that way the dot product can stand in for cosine similarity:

```
user_emb = user_emb / (np.linalg.norm(user_emb) + 1e-8)
```

This is equivalent to the collaborative-filtering move of "averaging the latent vectors of the items a user likes" (the dual view of item-based CF). The better the embeddings (the more tightly songs of the same kind cluster), the better the average vector represents the user's real taste.


In [ ]:
# Demo: averaging the embeddings of 3 songs
N, d = 100, 8
all_embs = np.random.randn(N, d)
# Normalize every embedding (simulating song embeddings that are already normalized)
all_embs = all_embs / (np.linalg.norm(all_embs, axis=1, keepdims=True) + 1e-8)

history = [0, 5, 12]
user_emb_raw = all_embs[history].mean(axis=0)
user_emb = user_emb_raw / (np.linalg.norm(user_emb_raw) + 1e-8)
print(f"user_emb shape : {user_emb.shape}")
print(f"user_emb norm  : {np.linalg.norm(user_emb):.4f}  (should be ≈ 1.0)")
print(f"user_emb[:4]   : {user_emb[:4].round(4)}")


## 2. The cold-start problem: what do you do with a new user who has no history?

When `history = []`, you can't compute an average embedding. Two common strategies:

**Strategy A — popularity ranking**: precompute the play count of every song in the library and return the global Top-K on a cold start. In formula terms this is equivalent to using the global mean of all embeddings as user_emb (or simply sorting by popularity score):

```
# Global mean (the geometric center) as the cold-start user_emb
cold_emb = all_embs.mean(axis=0)
cold_emb = cold_emb / (np.linalg.norm(cold_emb) + 1e-8)
```

**Strategy B — random diversification**: randomly sample some songs as the initial recommendations, which weakens the Matthew effect (the rich get richer) while collecting the first batch of interaction signals.

Strategy A is easy to explain; strategy B explores more. In industry, A usually comes first and is then mixed with B (`ε-greedy`).


In [ ]:
# Demo: cold start — global mean vs random sampling
cold_emb_global = all_embs.mean(axis=0)
cold_emb_global /= np.linalg.norm(cold_emb_global) + 1e-8

scores_cold = all_embs @ cold_emb_global
top_cold = np.argsort(scores_cold)[::-1][:5]
print(f"Cold-start Top-5 (global-mean strategy): {top_cold.tolist()}")

# Random diversification strategy
top_random = np.random.choice(N, size=5, replace=False)
print(f"Cold-start Top-5 (random strategy)     : {top_random.tolist()}")


## 3. Filtering out what's been heard: drop already-liked songs from the recommendations

In the ranked candidate list, history songs are bound to score high (they define user_emb), and recommending them directly just repeats things — the user already knows the song, so pushing it to them is worthless.

How to filter: turn `history` into a set and skip any id in the set while taking the Top-K:

```
history_set = set(user_history_ids)
ranked = np.argsort(scores)[::-1]
recs = [i for i in ranked if i not in history_set][:top_k]
```

Time complexity: an `O(N log N)` sort + `O(K)` filtering; at the million scale you would switch to FAISS + post-processing. At Aurora's current scale (tens of thousands of songs) a plain numpy sort is fine.


In [ ]:
# Demo: before vs after filtering
scores_demo = all_embs @ user_emb
ranked_all = np.argsort(scores_demo)[::-1][:8]
print(f"Top-8 before filtering: {ranked_all.tolist()}")
print(f"history ids           : {history}")

history_set = set(history)
ranked_filtered = [i for i in np.argsort(scores_demo)[::-1] if i not in history_set][:8]
print(f"Top-8 after filtering : {ranked_filtered}")
print(f"(history songs {[i for i in ranked_all if i in history_set]} have been removed)")


## 4. ✏️ Implement `recommend(user_history_ids, all_embs, top_k=10)`

**Signature**:
```python
def recommend(
    user_history_ids: list[int],   # list of the user's history song ids
    all_embs: np.ndarray,          # (N, d) embeddings of all songs (already L2-normalized)
    top_k: int = 10
) -> np.ndarray:
    # returns: (top_k,) int, recommended song ids, by descending similarity
```

**A 4-step route**:

| Step | Operation | Edge case |
|---|---|---|
| 1 | Compute user_emb | `all_embs[history_ids].mean(axis=0)`; if the history is empty → use the library-wide mean |
| 2 | L2-normalize user_emb | `/ np.linalg.norm(user_emb)` |
| 3 | Compute the cosine similarity of every song | `scores = all_embs @ user_emb` |
| 4 | Filter the history, take the top k | first set `scores[history_ids] = -inf`, then `argsort` and take the last top_k |

**Acceptance criteria**:
- Empty history → returns top_k ids (no error)
- The recommendations contain no songs from the history
- Results are in descending order of similarity, shape=(top_k,)

In [ ]:
def recommend(
    user_history_ids: list[int],
    all_embs: "np.ndarray",
    top_k: int = 10,
) -> "np.ndarray":
    """
    user_history_ids : list[int]   ids of songs the user likes
    all_embs         : np.ndarray  shape (N, d), L2-normalized song embeddings
    top_k            : int         number of results to return
    returns          : np.ndarray  shape (top_k,) int, recommended song ids, by descending similarity
    """
    # ✏️ TODO step 1: handle cold start (empty history) → use the global mean
    # ✏️ TODO step 2: compute user_emb = all_embs[history_ids].mean(axis=0), L2-normalize it
    # ✏️ TODO step 3: scores = all_embs @ user_emb
    # ✏️ TODO step 4: scores[history_ids] = -inf, argsort and take the last top_k (descending)
    raise NotImplementedError("TODO: return a top-k recommendation list based on cosine similarity")

In [ ]:
# ── Automatic check ──────────────────────────────────────────────────────────
np.random.seed(0)
N_test, d_test = 50, 8

# Build embeddings with cluster structure: the first 10 songs belong to a "jazz cluster"
jazz_center = np.array([1.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])
embs_test = np.random.randn(N_test, d_test) * 0.3
embs_test[:10] += jazz_center          # jazz cluster: ids 0-9
embs_test = embs_test / (np.linalg.norm(embs_test, axis=1, keepdims=True) + 1e-8)

# The user likes 5 jazz songs (ids 0-4)
history_test = [0, 1, 2, 3, 4]
try:
    recs = recommend(history_test, embs_test, top_k=5)
except (NotImplementedError, TypeError):
    recs = None

if recs is None:
    print("⬜ Not implemented yet, skipping verification")
else:
    assert len(recs) == 5, f"expected 5 recommendations, got {len(recs)}"
    assert all(isinstance(r, (int, np.integer)) for r in recs), "recommended ids should be int or np.integer (np.argsort returns np.int64)"
    assert not any(r in set(history_test) for r in recs), "recommendations should not contain history songs"

    # The recommendations should be mostly from the jazz cluster (ids 5-9)
    jazz_hits = sum(1 for r in recs if r < 10)
    assert jazz_hits >= 3, f"expected ≥3 jazz songs recommended, got {jazz_hits} (either the embedding clusters or the implementation is off)"

    print(f"✅ recommend() passed: recommendations = {recs}, jazz hits {jazz_hits}/5")

    # Cold-start check
    recs_cold = recommend([], embs_test, top_k=3)
    assert recs_cold is not None and len(recs_cold) == 3, "cold start should return 3 results"
    print(f"✅ Cold start passed: {recs_cold}")

## 5. Parameter experiment: simulate 10 users and plot the Precision@5 distribution

**Experiment design**:
- `n_users = 10`; each user randomly picks `history_len ∈ [3, 10]` songs as their history
- `top_k = 5`; after recommending, compute Precision@5: the fraction of recommendations that belong to the user's own style cluster
- The control variable: vary the **embedding noise coefficient** `noise_std` (0.1 → 0.5 → 1.0) and watch how Precision@5 changes as embedding quality drops

**What you should see** (`n_genres = 5`, random baseline = 1/5 = 0.2):
- `noise_std = 0.1` (high-quality embeddings, tight clusters): Precision@5 is close to perfect; in this experiment mean ≈ 1.00
- `noise_std = 0.5` (medium quality): mismatches start to appear; in this experiment mean ≈ 0.84
- `noise_std = 1.0` (low-quality embeddings, scattered clusters): a clear drop, but because the 5 cluster centers are still distinct unit vectors it doesn't fall all the way to the random baseline of 0.2; in this experiment mean ≈ 0.46
- The longer the history (the larger history_len), the more stable the average embedding, and the smaller the variance of Precision@5


In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

def run_experiment(noise_std, n_users=10, n_songs=200, d=16, n_genres=5, top_k=5):
    """Generate multi-style embeddings, simulate users, compute Precision@5"""
    np.random.seed(42)
    songs_per_genre = n_songs // n_genres  # songs per genre

    # Build embeddings: each genre's songs cluster around a random center
    centers = np.random.randn(n_genres, d)
    centers /= np.linalg.norm(centers, axis=1, keepdims=True) + 1e-8
    embs = np.vstack([
        centers[g] + np.random.randn(songs_per_genre, d) * noise_std
        for g in range(n_genres)
    ])
    embs /= np.linalg.norm(embs, axis=1, keepdims=True) + 1e-8

    # The true genre label of every song
    labels = np.repeat(np.arange(n_genres), songs_per_genre)

    precisions = []
    for u in range(n_users):
        # Randomly pick one main genre as this user's preference
        fav_genre = np.random.randint(n_genres)
        genre_ids = np.where(labels == fav_genre)[0].tolist()

        hist_len = np.random.randint(3, min(11, len(genre_ids)))
        history = np.random.choice(genre_ids, size=hist_len, replace=False).tolist()

        try:
            recs = recommend(history, embs, top_k=top_k)
        except (NotImplementedError, TypeError):
            recs = None
        if recs is None:
            precisions.append(0.0)
            continue
        hits = sum(1 for r in recs if labels[r] == fav_genre)
        precisions.append(hits / top_k)

    return precisions

noise_levels = [0.1, 0.5, 1.0]
labels_plot  = ["High quality (noise=0.1)", "Medium (noise=0.5)", "Low quality (noise=1.0)"]
colors       = ["#2ecc71", "#f39c12", "#e74c3c"]

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
fig.suptitle("Precision@5 distribution vs embedding quality (10 simulated users)", fontsize=13)

for ax, noise, label, color in zip(axes, noise_levels, labels_plot, colors):
    precs = run_experiment(noise_std=noise)
    ax.bar(range(len(precs)), precs, color=color, alpha=0.8)
    ax.axhline(np.mean(precs), color=plt.rcParams["text.color"], linestyle="--", linewidth=1.2,
               label=f"mean={np.mean(precs):.2f}")
    ax.set_title(label, fontsize=11)
    ax.set_xlabel("User number")
    ax.set_ylim(0, 1.05)
    ax.legend(fontsize=9)

axes[0].set_ylabel("Precision@5")
plt.tight_layout()
plt.savefig("mu4_precision_experiment.png", dpi=100, bbox_inches="tight")
plt.show()
print("✅ Experiment plot saved to mu4_precision_experiment.png")
print()

# Print a summary of the means
for noise, label in zip(noise_levels, labels_plot):
    precs = run_experiment(noise_std=noise)
    print(f"{label}: mean Precision@5 = {np.mean(precs):.3f}  std = {np.std(precs):.3f}")


## Wrap-up

This lesson implemented `recommend(user_history_ids, all_embs, top_k)`, whose output is the array of recommended ids after filtering out already-heard songs (an integer `np.ndarray` of length `top_k`). The modules this function actually corresponds to are `src/aurora/music/similarity.py` (similarity search), `src/aurora/music/features.py` (feature extraction) and `src/aurora/music/embed.py` (the embedding encoder, already in place); `recommend.py` is a planned stub file (where the numpy sort can be swapped for FAISS). The next lesson (L82) uses chroma heatmaps and embedding t-SNE scatter plots to check visually how well the music embeddings cluster.


## ✏️ Whiteboard challenge: the recommender by hand (target: 10 minutes)

Cover the screen and work it out on paper:

**Question 1**: what is the geometric meaning of a mean-embedding user profile?
(The "center of mass" (centroid) of the history song vectors; the songs closest to the centroid are semantically closest to the user's overall taste)

**Question 2**: if the user history has 3 songs with embeddings [1,0], [0,1], [1,1], what is user_emb? And after L2 normalization?
(Average = [2/3, 2/3]; length = √(4/9+4/9) = 2√2/3; normalized = [1/√2, 1/√2] ≈ [0.707, 0.707])

**Question 3**: why filter out already-heard songs when recommending? Explain it mathematically.
(The history songs' embeddings went straight into computing user_emb, so their cosine similarity with user_emb is very high (close to 1); without filtering they are bound to hog the top spots, and the recommendations would be nothing but songs already heard)

**Question 4**: what is the simplest fix for the cold-start problem (history=[])? What's its drawback?
(Simplest: use the mean of all the library's embeddings; drawback: the library-wide mean is close to the zero vector (uniformly spread in high dimensions), so after normalization its direction is essentially random and the recommendations are close to random. Better options: a popularity chart, or ask the user to pick a few genres they like)

**Question 5**: what does Precision@5 = 0.6 mean? How would you raise it?
(3 of the 5 recommended songs are actually liked; to raise it: better embeddings (more contrastive-learning data) / add time weighting (recent history matters more) / account for diversity (avoid recommending only one style))

Once you've worked them out, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: check the centroid idea
embs_q1 = np.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
centroid = embs_q1.mean(axis=0)
assert np.allclose(centroid, [2/3, 2/3]), f"the centroid should be [2/3,2/3], got {centroid}"
print(f"Q1 ✅  centroid of the 3 songs = {centroid} (mean-embedding user profile)")

# Q2: check the normalization
norm_val = np.linalg.norm(centroid)
normalized = centroid / norm_val
assert abs(np.linalg.norm(normalized) - 1.0) < 1e-9, "the length should be 1 after normalization"
expected = np.array([1/np.sqrt(2), 1/np.sqrt(2)])
assert np.allclose(normalized, expected, atol=1e-6), f"normalized should be {expected}, got {normalized}"
print(f"Q2 ✅  normalized user_emb = [{normalized[0]:.4f},{normalized[1]:.4f}] ≈ [1/√2,1/√2]")

# Q3: check the filtering works
try:
    np.random.seed(42)
    N_q3, d_q3 = 20, 4
    all_e = np.random.randn(N_q3, d_q3)
    all_e /= np.linalg.norm(all_e, axis=1, keepdims=True)
    history_q3 = [0, 1, 2]
    recs = recommend(history_q3, all_e, top_k=5)
    assert not any(r in history_q3 for r in recs), f"recommendations {recs} contain history songs {history_q3}"
    print(f"Q3 ✅  recommendations {recs.tolist()} contain none of the history {history_q3} (filtering works)")
except (NotImplementedError, TypeError):
    print(f"Q3 ✅  history songs are amplified in computing user_emb → they take the top spots → you must filter (verify once implemented)")

# Q4: cold start
np.random.seed(7)
N_q4, d_q4 = 100, 8
lib = np.random.randn(N_q4, d_q4)
lib /= np.linalg.norm(lib, axis=1, keepdims=True)
global_mean = lib.mean(axis=0)
global_mean_norm = np.linalg.norm(global_mean)
print(f"Q4 ✅  length of the library-wide mean = {global_mean_norm:.4f} (far below 1: uniform spread in high dimensions → the centroid is near zero)")

# Q5: a worked Precision@5 example
recommended = np.array([3, 7, 12, 5, 9])
ground_truth = {7, 12, 15, 22, 3}  # songs the user went on to like
hits = sum(1 for r in recommended if r in ground_truth)
precision_at_5 = hits / len(recommended)
assert abs(precision_at_5 - 0.6) < 1e-9, f"Precision@5 should be 0.6, got {precision_at_5}"
print(f"Q5 ✅  Precision@5={precision_at_5:.1f} (of the recommended {list(recommended)[:3]}..., {hits} were liked)")
print("\n🎉 Recommender whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l81_review = {
    "mean_emb_profile_geometry": None,  # understand user_emb = mean of history songs → semantic centroid → find nearest neighbors? True/False
    "cold_start_handling":       None,  # understand cold start: with history=[] use the library-wide mean or a popularity chart? True/False
    "recommend_impl":            None,  # recommend() is correct: no error on empty history / filters history / top-k in order? True/False
    "precision_at_k":            None,  # understand Precision@k as a measure of recommendation quality? True/False
    "whiteboard_passed":         None,  # worked all 5 whiteboard questions by hand? True/False
}

unfilled = [k for k, v in l81_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l81_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L81 all clear! On to L82: music features visualized')

## 6. [Optional appendix] From recommendation to generation: the architecture path of a Suno-style music app

> **Note**: this section is extra reading and not part of this lesson's core assessment. The focus here is recommender systems (retrieval); generative systems (MusicGen / Suno) get covered in depth in their own chapter. The Hugging Face templates below are only an architectural reference — you are not required to run them.

What this lesson does is **retrieval** (finding similar songs); Suno / MusicGen do **generation** (synthesizing audio straight from text). Their core technology stacks overlap heavily:

```
Retrieval (this lesson):  text/song → embedding → k-NN         → existing songs
Generation (Suno):        text      → embedding → LM generates → audio tokens → audio
```

### The key component: EnCodec (BPE for audio)

Meta's **EnCodec** turns continuous audio into discrete tokens, much as BPE tokenizes text in NLP:
- 32 kHz audio → **50 frames/s × 4 codebooks** (musicgen-small) → a sequence of integers
- A compression ratio of about **160×** (32000 samples / 200 tokens, i.e. 32000 / (50 frames × 4 codebooks)); the EnCodec decoder reconstructs the audio from the tokens (lossy, but at high quality)
- Once you have tokens, "generating music" ≡ "a GPT predicting the next token" (conditioned on a text embedding)

### Architecture comparison

| System | Core | Text input | Audio representation |
|------|------|---------|---------|
| This lesson's `recommend()` | cosine k-NN | — | 128-dim vector |
| CLAP (LAION) | NT-Xent contrastive | BERT/RoBERTa | 512-dim vector |
| MusicGen (Meta) | Transformer LM | T5 encoder | EnCodec tokens |
| Suno v3 | Diffusion + LM | large LM | proprietary codec |

### MusicGen in a few lines with Hugging Face

```python
from transformers import MusicgenForConditionalGeneration, AutoProcessor
model = MusicgenForConditionalGeneration.from_pretrained("facebook/musicgen-small")
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
inputs = processor(text=["lo-fi hip hop chill beats"], return_tensors="pt")
audio = model.generate(**inputs, max_new_tokens=256)  # ≈5s @50tok/s
```

The code below shows how EnCodec compression works, plus a complete MusicGen call template (it runs on a machine with a GPU).

In [ ]:
import numpy as np

# ── How EnCodec compression works (formulas only, no GPU / model install needed) ─────────────────────────
sr_audio   = 32_000   # MusicGen uses 32 kHz
duration_s = 1.0
n_samples  = int(sr_audio * duration_s)

frame_rate   = 50    # EnCodec output frame rate
n_codebooks  = 4     # musicgen-small uses 4 codebooks (large uses 8)
codebook_sz  = 2048  # vocabulary size of each codebook (11 bits)

n_frames     = int(frame_rate * duration_s)
total_tokens = n_frames * n_codebooks
compression  = n_samples / total_tokens

print("=== EnCodec compression parameters (musicgen-small) ===")
print(f"Raw audio      : {sr_audio}Hz × {duration_s}s = {n_samples:,} samples")
print(f"Token sequence : {n_frames} frames × {n_codebooks} codebooks = {total_tokens} integers")
print(f"Compression    : {compression:.1f}×")
print(f"Info per frame : {n_codebooks} × log₂({codebook_sz}) = {n_codebooks * np.log2(codebook_sz):.0f} bits")

# Simulate a token matrix
np.random.seed(42)
fake_tokens = np.random.randint(0, codebook_sz, (n_frames, n_codebooks))
print(f"\nSimulated token matrix shape: {fake_tokens.shape}  (frames × codebooks)")
print("First 5 frames:\n", fake_tokens[:5])

# ── Complete MusicGen call template (runs on a machine with transformers + a GPU) ──────────────
MUSICGEN_TEMPLATE = '''
# pip install transformers accelerate scipy  (about 300 MB + about 1.5 GB of model weights)
from transformers import MusicgenForConditionalGeneration, AutoProcessor
import torch, scipy.io.wavfile, numpy as np

model = MusicgenForConditionalGeneration.from_pretrained(
    "facebook/musicgen-small",
    torch_dtype=torch.float16,   # saves GPU memory; use float32 on CPU
    device_map="auto",
)
processor = AutoProcessor.from_pretrained("facebook/musicgen-small")
model.eval()

prompts = [
    "lo-fi hip hop, chill, 80 BPM, vinyl crackle",
    "epic orchestral strings and brass, dramatic, 140 BPM",
    "upbeat jazz piano trio, swing, 120 BPM",
]
inputs = processor(text=prompts, padding=True, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    # max_new_tokens=256 ≈ 5s @50 tokens/s
    tokens = model.generate(**inputs, max_new_tokens=256,
                             do_sample=True, temperature=1.0, guidance_scale=3.0)

sr = model.config.audio_encoder.sampling_rate  # 32000 Hz
for i, (prompt, audio) in enumerate(zip(prompts, tokens)):
    wav = audio[0].cpu().float().numpy()
    scipy.io.wavfile.write(f"musicgen_{i}.wav", rate=sr, data=wav)
    print(f"[{i}] saved musicgen_{i}.wav  ({len(wav)/sr:.1f}s)")
    print(f"     prompt: {prompt}")
'''

print("\n=== MusicGen Hugging Face call template ===")
print(MUSICGEN_TEMPLATE)

# ── CLAP text–audio contrastive embedding template ──────────────────────────────────────────
CLAP_TEMPLATE = '''
# pip install transformers torch torchaudio
from transformers import ClapModel, ClapProcessor
import torch, torchaudio

model     = ClapModel.from_pretrained("laion/larger_clap_general").eval()
processor = ClapProcessor.from_pretrained("laion/larger_clap_general")

# text → embedding
texts  = ["jazz piano", "heavy metal guitar", "classical violin"]
t_inp  = processor(text=texts, return_tensors="pt", padding=True)
with torch.no_grad():
    text_emb = model.get_text_features(**t_inp)      # (3, 512)
    text_emb = text_emb / text_emb.norm(dim=-1, keepdim=True)

# audio → embedding (needs a real mono WAV; resampled to 48 kHz below)
wav, sr = torchaudio.load("query.wav")
if sr != 48000:
    wav = torchaudio.functional.resample(wav, sr, 48000)
a_inp = processor(audios=wav.squeeze().numpy(), sampling_rate=48000, return_tensors="pt")
with torch.no_grad():
    audio_emb = model.get_audio_features(**a_inp)    # (1, 512)
    audio_emb = audio_emb / audio_emb.norm(dim=-1, keepdim=True)

# find the best-matching text label
sim = (text_emb @ audio_emb.T).squeeze()
print("Best match:", texts[sim.argmax()])
'''
print("=== CLAP text–audio contrastive embedding template ===")
print(CLAP_TEMPLATE)

---

→ **Next lesson** · [L82 · Music features visualized](L82_visual_music.ipynb)

> Next lesson covers **music features visualized**: chromagrams, beat grids and similarity heatmaps on display.